In [1]:
#import pygad
#import pyeasyga
#import pymoo
#import evopy


In [2]:
import math
import sys
import numpy as np
!{sys.executable} -m pip install matplotlib
import random
import matplotlib.pyplot as plt
import sys
!{sys.executable} -m pip install deap

import deap

!{sys.executable} -m pip install cellpylib
import cellpylib as cpl


!{sys.executable} -m pip install py_casim
from py_casim import Casim

#import multiprocessing
#from multiprocessing import Pool

from deap import base, creator, tools
from joblib import Parallel, delayed
import time
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for cellpylib: filename=cellpylib-2.4.0-py3-none-any.whl size=37922 sha256=11cd0a3eb1cba018c427c902a0821a172aca787a375494be0387922660fd690e
  Stored in directory: /root/.cache/pip/wheels/71/61/57/bbbbd5e8b79d6898242d075bd552bafab484034c3fcf710177
Successfully built cellpylib


In [3]:
#Estudiar cómo paralelizar!!

#Parámetros AG
NUM_STATES = 2
VEC_SIZE = 3
IND_SIZE = int(pow(NUM_STATES, 2*VEC_SIZE + 1))
POP_SIZE = 150

#Parámetros AC
CA_NUM = 100
CA_SIZE = 20
CA_TIMESTEPS = 40
CXPB, MUTPB, NGEN = 0.85, 0.03, 250

#Creador de fitness y de individuo
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

#Especificación de genes, individuos y poblacion
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


#Creador de regla de transición a partir de diccionario de reglas
def create_transition_rule(rule_dict):
    def my_rule(cells, r, t):
        # cells es un array con 2*r+1 elementos
        key = ''.join(str(int(c)) for c in cells)
        return rule_dict[key]
    return my_rule

#Función de fitness
def evaluate_CA_Majority(individual):
    densities = np.linspace(0.0, 1.0, CA_NUM)

    CAs = []


    for i in range(CA_NUM):
        numbers = np.random.rand(CA_SIZE)
        ca_binario = (numbers <= densities[i]).astype(int)

        CAs.append(np.expand_dims(ca_binario, axis=0))

    majority = [1 if np.sum(ca) >= CA_SIZE/2 else 0 for ca in CAs]

    #Diccionario de reglas
    rule_dict = {}
    for i in range(len(individual)):
        binario = format(i, f'0{2*VEC_SIZE + 1}b')
        rule_dict[binario] = individual[i]
    mi_regla = create_transition_rule(rule_dict)

    #Evolucionar autómatas con la regla con paralelización
    CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve)(CAs[i], timesteps=CA_TIMESTEPS,apply_rule=mi_regla,r=VEC_SIZE) for i in range(len(CAs)))

    #Estudiamos
    common_cells = []

    for i in range(len(CAs)):  # para cada autómata
        common_cells.append(np.sum(CAs[i][-1] == majority[i]))


    success = np.sum(np.array(common_cells) == CA_SIZE)


    return (success/CA_NUM,)

def evaluate_CA_Parity(individual):
    #Inicializar varios automatas con configuraciones iniciales aleatorias
    CAs = []
    CAs = [np.expand_dims(np.random.randint(2, size=CA_SIZE), axis=0) for _ in range(CA_NUM)]
    parity = [1 if (np.sum(ca) % 2 == 0) else 0 for ca in CAs]

    #Diccionario de reglas
    rule_dict = {}
    for i in range(len(individual)):
        binario = format(i, f'0{2*VEC_SIZE + 1}b')
        rule_dict[binario] = individual[i]
    mi_regla = create_transition_rule(rule_dict)

    #Evolucionar autómatas con la regla
    for i in range(len(CAs)):
        CAs[i] = cpl.evolve(CAs[i], timesteps=CA_TIMESTEPS,apply_rule=mi_regla,r=VEC_SIZE)


    common_cells = [np.sum(CAs[i][CA_TIMESTEPS - 1] == parity[i]) for i in range(len(CAs))]

    success = np.sum(np.array(common_cells) == CA_SIZE)

    return (success/CA_NUM,)


toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb = MUTPB)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate_CA_Majority)

def GA():
    pop = toolbox.population(n=POP_SIZE)


    # Evaluar toda la población
    #fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in pop)
    fitnesses = list(map(toolbox.evaluate, pop))
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit

    for g in range(NGEN):
        # Seleccionar la siguiente generación
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        # Cruce y mutación
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Evaluar individuos con fitness inválido
        #Ver condicion de parada de fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        pop[:] = offspring

        #top = tools.selBest(pop, 3)
        #if top[0].fitness.values[0] >= 0.8: #Estudiar condición de parada
            #print(f"Parado en la generación {g} con fitness {top[0].fitness.values[0]}")
            #return top

    return tools.selBest(pop, 3)


In [ ]:
top3 = GA()

for individual in top3:
    print(individual)
    print(individual.fitness.values)
    densities = np.linspace(0.0, 1.0, CA_NUM)
    CAs = []


    for i in range(CA_NUM):
        numbers = np.random.rand(CA_SIZE)
        ca_binario = (numbers <= densities[i]).astype(int)
        CAs.append(np.expand_dims(ca_binario, axis=0))

    majority = [1 if np.sum(ca) >= CA_SIZE/2 else 0 for ca in CAs]


    rule_dict = {}
    for i in range(len(individual)):
        binario = format(i, f'0{2*VEC_SIZE + 1}b')
        rule_dict[binario] = individual[i]


    mi_regla = create_transition_rule(rule_dict)

    #Evolucionarlos con la regla
    #Evolucionar y visualizar cada autómata
    CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve)(CAs[i], timesteps=CA_TIMESTEPS,apply_rule=mi_regla,r=VEC_SIZE) for i in range(len(CAs)))

    for i in range(len(CAs)):
        print(majority[i])
        plt.figure(figsize=(8, 4))
        plt.imshow(CAs[i], cmap='binary', interpolation='nearest', aspect='auto')
        plt.xlabel("Celda")
        plt.ylabel("Tiempo")
        plt.title(f"Evolución del autómata CA {i}")
        plt.show()


    #Estudiar variaciones de constantes y después realizar análisis con métricas de resultados. ¡Ver cómo guardar imágenes!
    '''PRUEBA 1:
    PARÁMETROS:
    VEC_SIZE = 3
    IND_SIZE = int(pow(2, 2*VEC_SIZE + 1))
    POP_SIZE = 100

    #Parámetros AC
    CA_NUM = 100
    CA_SIZE = 40
    CA_TIMESTEPS = 80
    CXPB, MUTPB, NGEN = 0.84, 0.01, 180
    RESULTADOS:
    Los mejores cromosomas, 70%, 65%, 63%
    Sobran pasos de evolución, suficiente con 50 o 60
    Cierto patrón a expandir zonas con densidad de cierto color hacia un lado (derecha o izquierda). Negro se ve mejor que blanco?
    Los que no consigue clasificar se queda intercambiando casilla blanca con casilla negra, y hay bastantes falsos positivos negros.
    '''